# MLB Clutch Metric — Historical + Live Jupyter Notebook

This notebook builds one row per plate appearance, identifies both hitter and pitcher, calculates context-adjusted run value, learns situation importance from Statcast win-expectancy changes, and produces hitter/pitcher situation and player summaries.

**Historical:** Retrosheet play-by-play (1908–2025). **Current season:** Baseball Savant Statcast. **Identity:** Chadwick Bureau Register. Raw observations are stored as Parquet; DuckDB is used for large-scale queries so the entire history does not have to fit into RAM.

In [2]:
%pip install -q pandas numpy pyarrow requests tqdm pybaseball duckdb


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Run once in a new Jupyter environment
%pip install -q pandas numpy pyarrow requests tqdm pybaseball duckdb

from pathlib import Path
from datetime import date, timedelta
import io, zipfile, requests
import numpy as np
import pandas as pd
from tqdm import tqdm

from pybaseball import statcast, chadwick_register
import duckdb

import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

Note: you may need to restart the kernel to use updated packages.


In [4]:


BASE = Path.cwd() / 'mlb_clutch_data'
PLAY_DIR = BASE / 'plays'
PA_DIR = BASE / 'pa'
AUX_DIR = BASE / 'aux'
SUMMARY_DIR = BASE / 'summaries'
for p in [PLAY_DIR, PA_DIR, AUX_DIR, SUMMARY_DIR]: p.mkdir(parents=True, exist_ok=True)

HISTORY_START = 1908
RETROSHEET_END = 2025
CURRENT_YEAR = date.today().year
LIVE_LOOKBACK_DAYS = 2
HISTORY_START_YEAR = 1908
HISTORY_END_YEAR = 2025


In [5]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*errors='ignore' is deprecated.*",
    category=FutureWarning
)

## 1. Why this is two data sources

Retrosheet is the historical backbone. Its parsed play-by-play contains the game ID, inning, inning half, score before the play, batter, pitcher, outs, pre/post runners, PA/AB flags, runs, results, and a pinch-hit flag. Statcast supplies modern pitch-level data and explicit run/win expectancy changes. Keep the source in a `data_source` column so results are auditable.

In [7]:
RETRO_URL = 'https://www.retrosheet.org/downloads/plays/{year}plays.zip'

NEEDED = [
    'gid','event','inning','top_bot','vis_home','batteam','pitteam',    'score_v','score_h','batter','pitcher','bathand','pithand',    'pa','ab','single','double','triple','hr','sh','sf','hbp','walk','k',    'iw','xi','roe','fc','othout','noout','outs_pre','outs_post',    'br1_pre','br2_pre','br3_pre','br1_post','br2_post','br3_post',    'runs','rbi','bat_f','pbp'
]

def download_retrosheet_year(year, overwrite=False):
    out = PLAY_DIR / f'retrosheet_{year}.parquet'
    if out.exists() and not overwrite:
        return out
    r = requests.get(RETRO_URL.format(year=year), timeout=180)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        names = [n for n in z.namelist() if n.lower().endswith('.csv')]
        if not names: raise RuntimeError(f'No CSV in {year} Retrosheet archive')
        name = max(names, key=lambda n: z.getinfo(n).file_size)
        with z.open(name) as f: df = pd.read_csv(f, low_memory=False)
    for c in NEEDED:
        if c not in df.columns: df[c] = pd.NA
    df = df[NEEDED].copy()
    df['season'] = year
    df['pbp_quality'] = df['pbp'].astype('string')
    for c in ['gid','batter','pitcher','br1_pre','br2_pre','br3_pre','br1_post','br2_post','br3_post']:
        df[c] = df[c].astype('string')
    df.to_parquet(out, index=False)
    return out

# Test first:
# download_retrosheet_year(2025)

# Full historical backfill — large download:
# for y in range(HISTORY_START, RETROSHEET_END + 1): download_retrosheet_year(y)


## 2. Player identity crosswalk

Retrosheet uses historical player IDs while Statcast uses MLBAM IDs. The Chadwick Register links these systems and provides a stable UUID-like identity. This lets career results be grouped across eras.

In [9]:
# ============================================================
# 3. CHADWICK REGISTER
# ============================================================

import pandas as pd
import requests
from io import BytesIO

CHADWICK_DIR = AUX_DIR / "chadwick"
CHADWICK_DIR.mkdir(parents=True, exist_ok=True)

CHADWICK_BASE = (
    "https://raw.githubusercontent.com/"
    "chadwickbureau/register/master/data/"
)


def load_chadwick_register(force_download=False):

    cache_file = CHADWICK_DIR / "people.parquet"

    if cache_file.exists() and not force_download:
        print("Loading cached Chadwick Register...")
        register = pd.read_parquet(cache_file)
        print(f"{len(register):,} people loaded.")
        return register

    parts = []

    for shard in "0123456789abcdef":

        url = CHADWICK_BASE + f"people-{shard}.csv"

        print(f"Downloading people-{shard}.csv ...")

        response = requests.get(
            url,
            timeout=180
        )

        response.raise_for_status()

        part = pd.read_csv(
            BytesIO(response.content),
            low_memory=False
        )

        parts.append(part)

    register = pd.concat(
        parts,
        ignore_index=True
    )

    required = [
        "key_uuid",
        "key_retro",
        "key_mlbam"
    ]

    missing = [
        column
        for column in required
        if column not in register.columns
    ]

    if missing:
        print("Missing columns:", missing)
        print("Available columns:")
        print(register.columns.tolist())

        raise ValueError(
            "Chadwick Register schema is missing "
            "one or more required identity fields."
        )

    for column in [
        "key_uuid",
        "key_person",
        "key_retro",
        "key_mlbam",
        "key_bbref",
        "key_fangraphs"
    ]:

        if column in register.columns:
            register[column] = (
                register[column]
                .astype("string")
            )

    register["player_name"] = (
        register["name_first"]
        .fillna("")
        .str.strip()
        + " "
        + register["name_last"]
        .fillna("")
        .str.strip()
    )

    register["player_name"] = (
        register["player_name"]
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.strip()
    )

    register.to_parquet(
        cache_file,
        index=False
    )

    print()
    print("================================")
    print("CHADWICK REGISTER LOADED")
    print("================================")
    print(f"People: {len(register):,}")
    print(f"Saved: {cache_file}")

    return register


register = load_chadwick_register(
    force_download=True
)

print(register.columns.tolist())


CHADWICK REGISTER LOADED
People: 526,894
Saved: C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_data\aux\chadwick\people.parquet
['key_person', 'key_uuid', 'key_mlbam', 'key_retro', 'key_bbref', 'key_bbref_minors', 'key_fangraphs', 'key_npb', 'key_sr_nfl', 'key_sr_nba', 'key_sr_nhl', 'key_wikidata', 'name_last', 'name_first', 'name_given', 'name_suffix', 'name_matrilineal', 'name_nick', 'birth_year', 'birth_month', 'birth_day', 'death_year', 'death_month', 'death_day', 'pro_played_first', 'pro_played_last', 'mlb_played_first', 'mlb_played_last', 'col_played_first', 'col_played_last', 'pro_managed_first', 'pro_managed_last', 'mlb_managed_first', 'mlb_managed_last', 'col_managed_first', 'col_managed_last', 'pro_umpired_first', 'pro_umpired_last', 'mlb_umpired_first', 'mlb_umpired_last', 'player_name']


In [10]:
# ============================================================
# CREATE PLAYER ID CROSSWALKS
# ============================================================

retro_map = (
    register[
        [
            "key_retro",
            "key_uuid",
            "key_mlbam",
            "player_name"
        ]
    ]
    .dropna(
        subset=["key_retro"]
    )
    .drop_duplicates(
        subset=["key_retro"]
    )
    .rename(
        columns={
            "key_retro": "retro_id",
            "key_uuid": "person_uuid",
            "key_mlbam": "mlbam_id"
        }
    )
)


mlbam_map = (
    register[
        [
            "key_mlbam",
            "key_uuid",
            "player_name"
        ]
    ]
    .dropna(
        subset=["key_mlbam"]
    )
    .drop_duplicates(
        subset=["key_mlbam"]
    )
    .rename(
        columns={
            "key_mlbam": "mlbam_id",
            "key_uuid": "person_uuid"
        }
    )
)


print(
    "Retrosheet IDs:",
    len(retro_map)
)

print(
    "MLBAM IDs:",
    len(mlbam_map)
)

Retrosheet IDs: 25799
MLBAM IDs: 135208


## 3. Historical RE24-style outcome value

For historical records, use the run-expectancy framework:

`run_value = runs_on_play + RE_after - RE_before`

This makes a single with runners on second/third and two outs different from a single with the bases empty. 

In [12]:
# ============================================================
# PREPARE RETROSHEET PLAYS FOR RE24
# ============================================================

def has_runner(series):
    """
    True when a Retrosheet runner ID exists.
    """
    s = series.astype("string")

    return (
        s.notna()
        & s.ne("")
        & s.ne("nan")
        & s.ne("<NA>")
    )


def prepare_historical_states(df):
    """
    Create all game-state variables needed for RE24.

    This function uses ALL plays in Retrosheet, not only PA rows.
    """

    df = df.copy()

    # --------------------------------------------------------
    # Basic numeric fields
    # --------------------------------------------------------

    df["inning"] = pd.to_numeric(
        df["inning"],
        errors="coerce"
    )

    df["top_bot"] = pd.to_numeric(
        df["top_bot"],
        errors="coerce"
    )

    df["score_v"] = pd.to_numeric(
        df["score_v"],
        errors="coerce"
    ).fillna(0)

    df["score_h"] = pd.to_numeric(
        df["score_h"],
        errors="coerce"
    ).fillna(0)

    df["outs_pre"] = pd.to_numeric(
        df["outs_pre"],
        errors="coerce"
    ).fillna(0).astype(int)

    df["outs_post"] = pd.to_numeric(
        df["outs_post"],
        errors="coerce"
    ).fillna(
        df["outs_pre"]
    ).astype(int)

    df["runs"] = pd.to_numeric(
        df["runs"],
        errors="coerce"
    ).fillna(0)

    # --------------------------------------------------------
    # Sort plays
    # --------------------------------------------------------

    if "pn" in df.columns:

        df["pn"] = pd.to_numeric(
            df["pn"],
            errors="coerce"
        )

        df = df.sort_values(
            ["gid", "pn"]
        ).copy()

    else:

        df = df.sort_values(
            ["gid", "inning", "top_bot"]
        ).copy()

    # --------------------------------------------------------
    # BASE STATE BEFORE PLAY
    #
    # 100 = runner on first
    # 010 = runner on second
    # 001 = runner on third
    # 110 = first + second
    # --------------------------------------------------------

    df["base_state_pre"] = (
        has_runner(df["br1_pre"])
        .astype(int)
        .astype(str)
        +
        has_runner(df["br2_pre"])
        .astype(int)
        .astype(str)
        +
        has_runner(df["br3_pre"])
        .astype(int)
        .astype(str)
    )

    # --------------------------------------------------------
    # BASE STATE AFTER PLAY
    # --------------------------------------------------------

    df["base_state_post"] = (
        has_runner(df["br1_post"])
        .astype(int)
        .astype(str)
        +
        has_runner(df["br2_post"])
        .astype(int)
        .astype(str)
        +
        has_runner(df["br3_post"])
        .astype(int)
        .astype(str)
    )

    # --------------------------------------------------------
    # HALF-INNING ID
    # --------------------------------------------------------

    df["half_id"] = (
        df["gid"].astype("string")
        + "_"
        + df["inning"].astype("Int64").astype(str)
        + "_"
        + df["top_bot"].astype("Int64").astype(str)
    )

    # --------------------------------------------------------
    # RUNS REMAINING IN HALF-INNING
    # --------------------------------------------------------

    df["runs_to_half_end"] = (
        df.groupby("half_id")["runs"]
        .transform(
            lambda x:
                x.iloc[::-1]
                 .cumsum()
                 .iloc[::-1]
        )
    )

    # Runs remaining after current play
    df["runs_after_play"] = (
        df["runs_to_half_end"]
        - df["runs"]
    )

    # --------------------------------------------------------
    # SCORE BEFORE PLAY
    # --------------------------------------------------------

    # top_bot = 0 -> away/visitor bats
    # top_bot = 1 -> home bats

    df["bat_score_before"] = np.where(
        df["top_bot"].eq(0),
        df["score_v"],
        df["score_h"]
    )

    df["fld_score_before"] = np.where(
        df["top_bot"].eq(0),
        df["score_h"],
        df["score_v"]
    )

    # --------------------------------------------------------
    # RUN DIFFERENTIAL FROM BATTER'S PERSPECTIVE
    # --------------------------------------------------------

    df["run_diff"] = (
        df["bat_score_before"]
        -
        df["fld_score_before"]
    )

    # --------------------------------------------------------
    # RISP
    # --------------------------------------------------------

    df["risp"] = (
        has_runner(df["br2_pre"])
        |
        has_runner(df["br3_pre"])
    )

    # --------------------------------------------------------
    # SCORE AFTER PLAY
    # --------------------------------------------------------

    df["away_score_after"] = (
        df["score_v"]
        +
        np.where(
            df["top_bot"].eq(0),
            df["runs"],
            0
        )
    )

    df["home_score_after"] = (
        df["score_h"]
        +
        np.where(
            df["top_bot"].eq(1),
            df["runs"],
            0
        )
    )

    # --------------------------------------------------------
    # IS THIS THE LAST PLAY OF THE GAME?
    # --------------------------------------------------------

    if "pn" in df.columns:

        final_play_number = (
            df.groupby("gid")["pn"]
            .transform("max")
        )

        df["is_game_end"] = (
            df["pn"]
            ==
            final_play_number
        )

    else:

        row_number = (
            df.groupby("gid")
            .cumcount()
        )

        total_rows = (
            df.groupby("gid")["gid"]
            .transform("size")
        )

        df["is_game_end"] = (
            row_number
            ==
            total_rows - 1
        )

    # --------------------------------------------------------
    # FINAL GAME SCORE
    # --------------------------------------------------------

    final_scores = (
        df.loc[
            df["is_game_end"],
            [
                "gid",
                "home_score_after",
                "away_score_after"
            ]
        ]
        .drop_duplicates(
            subset=["gid"]
        )
        .rename(
            columns={
                "home_score_after":
                    "final_home_score",

                "away_score_after":
                    "final_away_score"
            }
        )
    )

    df = df.merge(
        final_scores,
        on="gid",
        how="left"
    )

    # --------------------------------------------------------
    # HOME TEAM WIN
    # --------------------------------------------------------

    df["home_win"] = np.where(
        df["final_home_score"]
        >
        df["final_away_score"],
        1,

        np.where(
            df["final_home_score"]
            <
            df["final_away_score"],
            0,
            np.nan
        )
    )

    # --------------------------------------------------------
    # PRE-PLAY STATE KEY
    #
    # inning | score differential | half | outs | bases
    # --------------------------------------------------------

    score_diff_for_state = (
        df["run_diff"]
        .clip(-10, 10)
    )

    df["state_key_pre"] = (
        df["inning"]
        .astype("Int64")
        .astype(str)

        + "|"

        + score_diff_for_state
        .astype("Int64")
        .astype(str)

        + "|"

        + df["top_bot"]
        .astype("Int64")
        .astype(str)

        + "|"

        + df["outs_pre"]
        .astype("Int64")
        .astype(str)

        + "|"

        + df["base_state_pre"]
    )

    # --------------------------------------------------------
    # STATE AFTER PLAY
    # --------------------------------------------------------

    df["inning_post"] = df["inning"]

    df["top_bot_post"] = df["top_bot"]

    df["outs_post_state"] = df["outs_post"]

    df["base_state_post_state"] = (
        df["base_state_post"]
    )

    inning_finished = (
        df["outs_post"] >= 3
    )

    active_game = (
        ~df["is_game_end"]
    )

    # Top half -> bottom half
    top_to_bottom = (
        inning_finished
        &
        active_game
        &
        df["top_bot"].eq(0)
    )

    # Bottom half -> top of next inning
    bottom_to_top = (
        inning_finished
        &
        active_game
        &
        df["top_bot"].eq(1)
    )

    df.loc[
        top_to_bottom,
        "top_bot_post"
    ] = 1

    df.loc[
        bottom_to_top,
        "top_bot_post"
    ] = 0

    df.loc[
        bottom_to_top,
        "inning_post"
    ] = (
        df.loc[
            bottom_to_top,
            "inning"
        ]
        + 1
    )

    # New half inning begins empty
    reset_state = (
        inning_finished
        &
        active_game
    )

    df.loc[
        reset_state,
        "outs_post_state"
    ] = 0

    df.loc[
        reset_state,
        "base_state_post_state"
    ] = "000"

    # --------------------------------------------------------
    # SCORE DIFFERENTIAL AFTER PLAY
    # --------------------------------------------------------

    df["bat_score_after"] = np.where(
        df["top_bot_post"].eq(0),
        df["away_score_after"],
        df["home_score_after"]
    )

    df["fld_score_after"] = np.where(
        df["top_bot_post"].eq(0),
        df["home_score_after"],
        df["away_score_after"]
    )

    df["run_diff_post"] = (
        df["bat_score_after"]
        -
        df["fld_score_after"]
    )

    # --------------------------------------------------------
    # POST-PLAY STATE KEY
    # --------------------------------------------------------

    df["state_key_post"] = (
        df["inning_post"]
        .astype("Int64")
        .astype(str)

        + "|"

        + df["run_diff_post"]
        .clip(-10, 10)
        .astype("Int64")
        .astype(str)

        + "|"

        + df["top_bot_post"]
        .astype("Int64")
        .astype(str)

        + "|"

        + df["outs_post_state"]
        .astype("Int64")
        .astype(str)

        + "|"

        + df["base_state_post_state"]
    )

    return df

In [13]:
def build_retable(
    start_year=HISTORY_START_YEAR,
    end_year=HISTORY_END_YEAR
):
    """
    Build the historical Run Expectancy (RE24) table
    from all Retrosheet play-by-play data.
    """

    pre_parts = []
    post_parts = []

    for year in tqdm(
        range(start_year, end_year + 1),
        desc="Building RE24",
        unit="season"
    ):

        print(f"\nProcessing {year}...")

        path = PLAY_DIR / f"retrosheet_{year}.parquet"

        # Download the season if it isn't already present
        if not path.exists():
            download_retrosheet_year(year)

        # Load ALL plays for the season
        df = pd.read_parquet(path)

        # Create game-state information
        df = prepare_historical_states(df)

        # -----------------------------
        # BEFORE-PLAY RUN EXPECTANCY
        # -----------------------------

        pre = (
            df.groupby("state_key_pre")["runs_to_half_end"]
            .agg(
                run_sum="sum",
                n="count"
            )
            .reset_index()
        )

        pre = pre.rename(
            columns={
                "state_key_pre": "state_key"
            }
        )

        # -----------------------------
        # AFTER-PLAY RUN EXPECTANCY
        # -----------------------------

        post = (
            df.groupby("state_key_post")["runs_after_play"]
            .agg(
                run_sum="sum",
                n="count"
            )
            .reset_index()
        )

        post = post.rename(
            columns={
                "state_key_post": "state_key"
            }
        )

        pre_parts.append(pre)
        post_parts.append(post)

    # Combine all seasons
    pre_all = pd.concat(
        pre_parts,
        ignore_index=True
    )

    post_all = pd.concat(
        post_parts,
        ignore_index=True
    )

    # Aggregate across all seasons
    pre = (
        pre_all
        .groupby("state_key")[
            ["run_sum", "n"]
        ]
        .sum()
        .reset_index()
    )

    post = (
        post_all
        .groupby("state_key")[
            ["run_sum", "n"]
        ]
        .sum()
        .reset_index()
    )

    # Calculate expected runs
    pre["RE_before"] = (
        pre["run_sum"] /
        pre["n"]
    )

    post["RE_after"] = (
        post["run_sum"] /
        post["n"]
    )

    # Combine
    re_table = (
        pre[
            ["state_key", "RE_before"]
        ]
        .merge(
            post[
                ["state_key", "RE_after"]
            ],
            on="state_key",
            how="outer"
        )
    )

    # Save the table
    output_file = (
        AUX_DIR /
        "re24_table.parquet"
    )

    re_table.to_parquet(
        output_file,
        index=False
    )

    print("\n================================")
    print("RE24 TABLE COMPLETE")
    print("================================")
    print(
        f"Number of states: "
        f"{len(re_table):,}"
    )
    print(
        f"Saved to: "
        f"{output_file}"
    )

    return re_table

In [14]:
re_table = build_retable()

Building RE24:   0%|                                                                  | 0/118 [00:00<?, ?season/s]


Processing 1908...


Building RE24:   1%|▍                                                         | 1/118 [00:06<13:14,  6.79s/season]


Processing 1909...


Building RE24:   2%|▉                                                         | 2/118 [00:13<12:41,  6.56s/season]

...


Processing 2024...


Building RE24:  99%|███████████████████████████████████████████████████████▌| 117/118 [16:28<00:10, 10.77s/season]


Processing 2025...


Building RE24: 100%|████████████████████████████████████████████████████████| 118/118 [16:39<00:00,  8.47s/season]



RE24 TABLE COMPLETE
Number of states: 11,863
Saved to: C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_data\aux\re24_table.parquet


## 4. Situation importance instead of arbitrary weights

Do not hard-code something like '9th inning = 2x'. Instead, use Statcast's `delta_home_win_exp` to learn how much a situation typically moves win expectancy. At the PA level, sum pitch-level win-expectancy changes. Then calculate the mean absolute PA-level change for each situation bucket and normalize it to 1.0.

In [16]:
def inning_band(x):
    return pd.cut(pd.to_numeric(x, errors='coerce'), [0,3,6,9,np.inf], labels=['1-3','4-6','7-9','10+']).astype('string')

def score_band(x):
    return pd.cut(pd.to_numeric(x, errors='coerce').fillna(0), [-np.inf,-4,-3,-2,-1,0,1,2,3,4,np.inf], labels=['-4+','-3','-2','-1','0','+1','+2','+3','+4','+5+']).astype('string')

def add_situation(df):
    df = df.copy()
    df['base_state'] = flag(df.br1_pre).astype(int).astype(str) + flag(df.br2_pre).astype(int).astype(str) + flag(df.br3_pre).astype(int).astype(str)
    df['bat_score'] = np.where(df.top_bot.eq(1), df.score_h, df.score_v)
    df['fld_score'] = np.where(df.top_bot.eq(1), df.score_v, df.score_h)
    df['run_diff'] = pd.to_numeric(df.bat_score, errors='coerce') - pd.to_numeric(df.fld_score, errors='coerce')
    df['risp'] = flag(df.br2_pre) | flag(df.br3_pre)
    df['inning_band'] = inning_band(df.inning)
    df['score_band'] = score_band(df.run_diff)
    df['situation_key'] = df.inning_band.astype(str) + '|' + df.score_band.astype(str) + '|' + df.top_bot.astype(str) + '|' + df.outs_pre.astype('Int64').astype(str) + '|' + df.base_state.astype(str)
    return df

def build_leverage_table(start_date='2024-03-20', end_date='2025-11-01', force=False):
    target = AUX_DIR / 'leverage_index.parquet'
    if target.exists() and not force: return pd.read_parquet(target)
    sc = statcast(start_dt=start_date, end_dt=end_date, verbose=True, parallel=True)
    sc = sc.sort_values(['game_pk','at_bat_number','pitch_number']).copy()
    sc['pa_id'] = sc.game_pk.astype('string') + '_' + sc.at_bat_number.astype('string')
    first = sc.groupby('pa_id', as_index=False).first()
    wpa = sc.groupby('pa_id', as_index=False)['delta_home_win_exp'].sum().rename(columns={'delta_home_win_exp':'pa_wpa'})
    pa = first.merge(wpa, on='pa_id', how='left')
    pa = pa.rename(columns={'inning_topbot':'top_bot','outs_when_up':'outs_pre','on_1b':'br1_pre','on_2b':'br2_pre','on_3b':'br3_pre','home_score':'score_h','away_score':'score_v'})
    pa = add_situation(pa)
    pa['abs_pa_wpa'] = pd.to_numeric(pa.pa_wpa, errors='coerce').abs()
    li = pa.groupby('situation_key').abs_pa_wpa.agg(mean_abs_wpa='mean', pa_count='size').reset_index()
    li['leverage_index'] = li.mean_abs_wpa / pa.abs_pa_wpa.mean()
    li.loc[li.pa_count < 250, 'leverage_index'] = 1.0
    li.to_parquet(target, index=False)
    return li

# leverage = build_leverage_table()


In [17]:
leverage = build_leverage_table()

## 5. Make one PA row per historical at-bat/plate appearance

The raw Retrosheet file contains non-PA events such as steals and wild pitches. Keep them for RE24 calculation, but the clutch dataset itself should contain only `pa == 1` observations.

In [19]:
EVENTS = [('hr','home_run'),('triple','triple'),('double','double'),('single','single'),('iw','intentional_walk'),('walk','walk'),('hbp','hit_by_pitch'),('k','strikeout'),('fc','fielders_choice'),('roe','reach_on_error'),('sf','sac_fly'),('sh','sac_bunt'),('xi','interference'),('othout','other_out'),('noout','other_pa')]

def pa_result(df):
    out = pd.Series(pd.NA, index=df.index, dtype='string')
    for col, label in EVENTS:
        if col in df.columns:
            m = pd.to_numeric(df[col], errors='coerce').fillna(0).eq(1) & out.isna()
            out.loc[m] = label
    return out

def build_historical_pa(year, re_table, leverage_table):
    df = pd.read_parquet(PLAY_DIR / f'retrosheet_{year}.parquet')
    df = df[pd.to_numeric(df.pa, errors='coerce').fillna(0).eq(1)].copy()
    df = states(df)
    df = add_situation(df)
    re_before = re_table.set_index('state').RE_before
    re_after = re_table.set_index('state').RE_after
    df['run_value'] = pd.to_numeric(df.runs, errors='coerce').fillna(0) + df.state_post.map(re_after).fillna(0) - df.state_pre.map(re_before).fillna(0)
    df['pa_result'] = pa_result(df)
    df['is_pinch_hitter'] = pd.to_numeric(df.bat_f, errors='coerce').eq(11)
    df['game_id'] = df.season.astype(str) + '_' + df.gid.astype(str)
    df['data_source'] = 'Retrosheet'
    df = df.merge(retro_map.rename(columns={'key_retro':'batter','key_uuid':'batter_uuid','key_mlbam':'batter_mlbam','player_name':'batter_name'}), on='batter', how='left')
    df = df.merge(retro_map.rename(columns={'key_retro':'pitcher','key_uuid':'pitcher_uuid','key_mlbam':'pitcher_mlbam','player_name':'pitcher_name'}), on='pitcher', how='left')
    df = df.merge(leverage_table[['situation_key','leverage_index']], on='situation_key', how='left')
    df['leverage_index'] = df.leverage_index.fillna(1.0)
    keep = ['season','game_id','gid','inning','top_bot','outs_pre','score_v','score_h','bat_score','fld_score','run_diff','batter_uuid','batter_mlbam','batter_name','pitcher_uuid','pitcher_mlbam','pitcher_name','br1_pre','br2_pre','br3_pre','base_state','risp','is_pinch_hitter','pa_result','event','runs','rbi','RE_before','RE_after','run_value','inning_band','score_band','situation_key','leverage_index','pbp_quality','data_source']
    for c in keep:
        if c not in df.columns: df[c] = pd.NA
    return df[keep]


## 6. Current-season updater

Statcast exposes pre-pitch inning, outs, score, runners, batter, pitcher, result, and `delta_run_exp`/`delta_home_win_exp`. The updater re-downloads the last two days so corrections are incorporated rather than simply appended.

In [21]:
def normalize_mlbam_id(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64").astype("string")

# ============================================================
# CURRENT SEASON UPDATER
# NO DEPENDENCY ON flag()
# ============================================================

def update_current_season():

    target_dir = (
        PA_DIR
        / f"season={CURRENT_YEAR}"
    )

    target_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    target = (
        target_dir
        / f"pa_{CURRENT_YEAR}.parquet"
    )

    # --------------------------------------------------------
    # LOAD EXISTING DATA
    # --------------------------------------------------------

    if target.exists():

        old = pd.read_parquet(target)

        print(
            f"Existing current-season PAs: "
            f"{len(old):,}"
        )

        if (
            "game_date" in old.columns
            and not old.empty
        ):

            last_date = (
                pd.to_datetime(
                    old["game_date"],
                    errors="coerce"
                )
                .dropna()
                .max()
                .date()
            )

        else:

            last_date = CURRENT_SEASON_START

        fetch_start = max(
            CURRENT_SEASON_START,
            last_date - timedelta(
                days=LIVE_LOOKBACK_DAYS
            )
        )

    else:

        old = pd.DataFrame()

        fetch_start = CURRENT_SEASON_START

    fetch_end = date.today()

    print(
        f"Refreshing Statcast "
        f"{fetch_start} -> {fetch_end}"
    )

    # --------------------------------------------------------
    # DOWNLOAD CURRENT STATCAST DATA
    # --------------------------------------------------------

    new = fetch_statcast_pa(
        fetch_start,
        fetch_end
    )

    if new.empty:

        print(
            "No new completed PAs found."
        )

        return old

    # --------------------------------------------------------
    # ADD BATTER/PITCHER NAMES
    # --------------------------------------------------------

    new = add_statcast_player_names(
        new
    )

    # --------------------------------------------------------
    # LEVERAGE
    # --------------------------------------------------------

    leverage_file = (
        AUX_DIR
        / "leverage_index.parquet"
    )

    if leverage_file.exists():

        leverage = pd.read_parquet(
            leverage_file
        )

        new = new.merge(
            leverage[
                [
                    "situation_key",
                    "leverage_index"
                ]
            ],
            on="situation_key",
            how="left"
        )

        new["leverage_index"] = (
            pd.to_numeric(
                new["leverage_index"],
                errors="coerce"
            )
            .fillna(1.0)
        )

    else:

        print(
            "WARNING: leverage_index.parquet "
            "does not exist."
        )

        new["leverage_index"] = 1.0

    # --------------------------------------------------------
    # PINCH HITTERS
    # --------------------------------------------------------

    print(
        "Checking pinch hitters..."
    )

    new = enrich_current_pinch_hitters(
        new,
        new["game_pk"]
        .dropna()
        .unique()
    )

    # --------------------------------------------------------
    # STANDARDIZE IDS
    # --------------------------------------------------------

    new["game_id"] = (
        new["game_pk"]
        .astype("string")
    )

    new["batter_mlbam"] = (
        pd.to_numeric(
            new["batter_mlbam"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    new["pitcher_mlbam"] = (
        pd.to_numeric(
            new["pitcher_mlbam"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    # --------------------------------------------------------
    # UNIQUE PA KEY
    # --------------------------------------------------------

    new["pa_key"] = (
        new["game_id"]
        + "_"
        + new["at_bat_number"]
        .astype("string")
    )

    # --------------------------------------------------------
    # REMOVE OLD OVERLAPPING PAs
    # --------------------------------------------------------

    if not old.empty:

        old["pa_key"] = (
            old["game_id"]
            .astype("string")
            + "_"
            + old["at_bat_number"]
            .astype("string")
        )

        new_keys = set(
            new["pa_key"]
        )

        old = old[
            ~old["pa_key"]
            .isin(new_keys)
        ].copy()

    # --------------------------------------------------------
    # COMBINE OLD + NEW
    # --------------------------------------------------------

    combined = pd.concat(
        [
            old,
            new
        ],
        ignore_index=True,
        sort=False
    )

    # --------------------------------------------------------
    # REMOVE DUPLICATES
    # --------------------------------------------------------

    combined = (
        combined
        .sort_values(
            [
                "game_date",
                "game_id",
                "at_bat_number"
            ]
        )
        .drop_duplicates(
            subset=[
                "game_id",
                "at_bat_number"
            ],
            keep="last"
        )
    )

    combined = combined.drop(
        columns=["pa_key"],
        errors="ignore"
    )

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    combined.to_parquet(
        target,
        index=False
    )

    print()
    print(
        "========================================"
    )
    print(
        "CURRENT SEASON UPDATED"
    )
    print(
        "========================================"
    )

    print(
        f"Total PAs: {len(combined):,}"
    )

    print(
        f"First date: "
        f"{combined['game_date'].min()}"
    )

    print(
        f"Last date: "
        f"{combined['game_date'].max()}"
    )

    print(
        f"Saved: {target}"
    )

    return combined


In [22]:
# ============================================================
# ROBUST STATCAST DOWNLOADER
# ============================================================


STATCAST_TIMEOUT = 120
STATCAST_RETRIES = 4
STATCAST_RETRY_WAIT = 10


def fetch_statcast_chunk(
    start_date,
    end_date,
    retries=STATCAST_RETRIES
):
    """
    Download a small Statcast date range with retries.

    Keeping ranges small reduces the chance of Baseball Savant
    timing out or hitting its query-size limitations.
    """

    for attempt in range(1, retries + 1):

        try:

            print(
                f"  Request "
                f"{start_date} -> {end_date} "
                f"(attempt {attempt}/{retries})"
            )

            df = statcast(
                start_dt=str(start_date),
                end_dt=str(end_date),
                verbose=False,
                parallel=False
            )

            if df is None:
                return pd.DataFrame()

            return df

        except Exception as e:

            print(
                f"  Request failed: {type(e).__name__}: {e}"
            )

            if attempt < retries:

                wait = (
                    STATCAST_RETRY_WAIT
                    * attempt
                )

                print(
                    f"  Retrying in {wait} seconds..."
                )

                time.sleep(wait)

            else:

                print(
                    "  Giving up on this chunk."
                )

                return pd.DataFrame()


# ============================================================
# FIXED STATCAST -> ONE ROW PER PLATE APPEARANCE
# ============================================================


In [23]:
# ============================================================
# STATCAST -> ONE ROW PER PLATE APPEARANCE
# No dependency on flag()
# ============================================================

def fetch_statcast_pa(start_date, end_date):

    print(
        f"Downloading Statcast: "
        f"{start_date} -> {end_date}"
    )

    sc = statcast(
        start_dt=str(start_date),
        end_dt=str(end_date),
        verbose=True,
        parallel=False
    )

    if sc is None or sc.empty:
        print("No Statcast data returned.")
        return pd.DataFrame()

    print(
        f"Downloaded {len(sc):,} pitch rows."
    )

    # --------------------------------------------------------
    # Make sure all required columns exist
    # --------------------------------------------------------

    required = [
        "game_date",
        "game_pk",
        "at_bat_number",
        "pitch_number",
        "batter",
        "pitcher",
        "events",
        "inning",
        "inning_topbot",
        "outs_when_up",
        "on_1b",
        "on_2b",
        "on_3b",
        "home_score",
        "away_score",
        "bat_score",
        "fld_score",
        "delta_run_exp",
        "delta_home_win_exp"
    ]

    missing = [
        c for c in required
        if c not in sc.columns
    ]

    if missing:
        raise KeyError(
            f"Statcast is missing: {missing}"
        )

    # --------------------------------------------------------
    # Sort pitches
    # --------------------------------------------------------

    sc = sc.sort_values(
        [
            "game_pk",
            "at_bat_number",
            "pitch_number"
        ]
    ).copy()

    # --------------------------------------------------------
    # Completed PA
    # --------------------------------------------------------

    completed = sc[
        sc["events"].notna()
    ].copy()

    if completed.empty:
        print(
            "No completed plate appearances."
        )
        return pd.DataFrame()

    # --------------------------------------------------------
    # First pitch = PRE-PA situation
    # --------------------------------------------------------

    pre_columns = [
        "game_date",
        "game_pk",
        "at_bat_number",
        "pitch_number",
        "batter",
        "pitcher",
        "inning",
        "inning_topbot",
        "outs_when_up",
        "on_1b",
        "on_2b",
        "on_3b",
        "home_score",
        "away_score",
        "bat_score",
        "fld_score"
    ]

    first_pitch = (
        sc[pre_columns]
        .groupby(
            [
                "game_pk",
                "at_bat_number"
            ],
            as_index=False
        )
        .first()
    )

    # --------------------------------------------------------
    # PA result + PA-level RE/WPA
    # --------------------------------------------------------

    pa_outcome = (
        completed
        .groupby(
            [
                "game_pk",
                "at_bat_number"
            ],
            as_index=False
        )
        .agg(
            pa_result=(
                "events",
                "last"
            ),

            pa_description=(
                "des",
                "last"
            ),

            delta_run_exp=(
                "delta_run_exp",
                "sum"
            ),

            delta_home_win_exp=(
                "delta_home_win_exp",
                "sum"
            )
        )
    )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    pa = first_pitch.merge(
        pa_outcome,
        on=[
            "game_pk",
            "at_bat_number"
        ],
        how="inner"
    )

    # --------------------------------------------------------
    # DATE
    # --------------------------------------------------------

    pa["game_date"] = pd.to_datetime(
        pa["game_date"],
        errors="coerce"
    )

    pa["season"] = (
        pa["game_date"].dt.year
    )

    # --------------------------------------------------------
    # BATTER / PITCHER IDs
    # --------------------------------------------------------

    pa["batter_mlbam"] = (
        pd.to_numeric(
            pa["batter"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    pa["pitcher_mlbam"] = (
        pd.to_numeric(
            pa["pitcher"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    # --------------------------------------------------------
    # INNING / HALF
    # --------------------------------------------------------

    pa["inning"] = (
        pd.to_numeric(
            pa["inning"],
            errors="coerce"
        )
        .astype("Int64")
    )

    pa["top_bot"] = np.where(
        pa["inning_topbot"]
        .astype("string")
        .str.lower()
        .eq("top"),
        0,
        1
    )

    # --------------------------------------------------------
    # OUTS
    # --------------------------------------------------------

    pa["outs_pre"] = (
        pd.to_numeric(
            pa["outs_when_up"],
            errors="coerce"
        )
        .astype("Int64")
    )

    # --------------------------------------------------------
    # SCORES
    # --------------------------------------------------------

    pa["score_h"] = pd.to_numeric(
        pa["home_score"],
        errors="coerce"
    )

    pa["score_v"] = pd.to_numeric(
        pa["away_score"],
        errors="coerce"
    )

    pa["bat_score_before"] = pd.to_numeric(
        pa["bat_score"],
        errors="coerce"
    )

    pa["fld_score_before"] = pd.to_numeric(
        pa["fld_score"],
        errors="coerce"
    )

    pa["run_diff"] = (
        pa["bat_score_before"]
        -
        pa["fld_score_before"]
    )

    # --------------------------------------------------------
    # RUNNERS
    # --------------------------------------------------------

    pa["br1_pre"] = (
        pd.to_numeric(
            pa["on_1b"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    pa["br2_pre"] = (
        pd.to_numeric(
            pa["on_2b"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    pa["br3_pre"] = (
        pd.to_numeric(
            pa["on_3b"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    # --------------------------------------------------------
    # BASE STATE
    # --------------------------------------------------------

    runner_1 = pa["br1_pre"].notna().astype(int)
    runner_2 = pa["br2_pre"].notna().astype(int)
    runner_3 = pa["br3_pre"].notna().astype(int)

    pa["base_state"] = (
        runner_1.astype(str)
        + runner_2.astype(str)
        + runner_3.astype(str)
    )

    # --------------------------------------------------------
    # RISP
    # --------------------------------------------------------

    pa["risp"] = (
        runner_2.astype(bool)
        |
        runner_3.astype(bool)
    )

    # --------------------------------------------------------
    # INNING BUCKET
    # --------------------------------------------------------

    pa["inning_bucket"] = np.where(
        pa["inning"] >= 10,
        "10+",
        pa["inning"].astype(str)
    )

    # --------------------------------------------------------
    # SCORE DIFFERENTIAL BUCKET
    # --------------------------------------------------------

    pa["score_diff_clamped"] = (
        pa["run_diff"]
        .clip(-10, 10)
        .round()
        .astype("Int64")
    )

    # --------------------------------------------------------
    # SITUATION KEY
    # --------------------------------------------------------

    pa["situation_key"] = (
        pa["inning_bucket"].astype(str)
        + "|"
        + pa["score_diff_clamped"].astype(str)
        + "|"
        + pa["top_bot"].astype(str)
        + "|"
        + pa["outs_pre"].astype(str)
        + "|"
        + pa["base_state"].astype(str)
    )

    # --------------------------------------------------------
    # RUN / WIN EXPECTANCY VALUES
    # --------------------------------------------------------

    pa["run_value"] = pd.to_numeric(
        pa["delta_run_exp"],
        errors="coerce"
    )

    pa["pa_wpa"] = pd.to_numeric(
        pa["delta_home_win_exp"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # IDENTIFIERS
    # --------------------------------------------------------

    pa["game_id"] = (
        pa["game_pk"]
        .astype("string")
    )

    pa["data_source"] = "Statcast"

    pa["pbp_quality"] = "full_Statcast"

    return pa

In [24]:
# ============================================================
# ADD BATTER AND PITCHER NAMES
# ============================================================

def add_statcast_player_names(pa):

    pa = pa.copy()

    # --------------------------------------------------------
    # Batter lookup
    # --------------------------------------------------------

    batter_lookup = (
        mlbam_map[
            [
                "mlbam_id",
                "person_uuid",
                "player_name"
            ]
        ]
        .copy()
        .rename(
            columns={
                "mlbam_id": "batter_mlbam",
                "person_uuid": "batter_uuid",
                "player_name": "batter_name"
            }
        )
    )

    # --------------------------------------------------------
    # Pitcher lookup
    # --------------------------------------------------------

    pitcher_lookup = (
        mlbam_map[
            [
                "mlbam_id",
                "person_uuid",
                "player_name"
            ]
        ]
        .copy()
        .rename(
            columns={
                "mlbam_id": "pitcher_mlbam",
                "person_uuid": "pitcher_uuid",
                "player_name": "pitcher_name"
            }
        )
    )

    # --------------------------------------------------------
    # Make sure IDs have the same type
    # --------------------------------------------------------

    pa["batter_mlbam"] = (
        pa["batter_mlbam"]
        .astype("string")
    )

    pa["pitcher_mlbam"] = (
        pa["pitcher_mlbam"]
        .astype("string")
    )

    batter_lookup["batter_mlbam"] = (
        batter_lookup["batter_mlbam"]
        .astype("string")
    )

    pitcher_lookup["pitcher_mlbam"] = (
        pitcher_lookup["pitcher_mlbam"]
        .astype("string")
    )

    # --------------------------------------------------------
    # Merge batter
    # --------------------------------------------------------

    pa = pa.merge(
        batter_lookup,
        on="batter_mlbam",
        how="left"
    )

    # --------------------------------------------------------
    # Merge pitcher
    # --------------------------------------------------------

    pa = pa.merge(
        pitcher_lookup,
        on="pitcher_mlbam",
        how="left"
    )

    return pa

In [25]:
# ============================================================
# PINCH-HITTER ENRICHMENT
# ============================================================

def enrich_current_pinch_hitters(pa, game_pks, request_pause=0.10):
    """
    Identify pinch hitters in the supplied Statcast PA data
    using MLB StatsAPI boxscores.

    A player is marked as a pinch hitter when the boxscore
    identifies their position as PH / position code 11.
    """

    pa = pa.copy()

    pinch_hitter_ids = set()

    for game_pk in game_pks:

        url = (
            "https://statsapi.mlb.com/api/v1/game/"
            f"{int(game_pk)}/boxscore"
        )

        try:
            response = requests.get(
                url,
                timeout=60
            )

            response.raise_for_status()

            data = response.json()

            teams = data.get(
                "teams",
                {}
            )

            for side in ["away", "home"]:

                team = teams.get(
                    side,
                    {}
                )

                players = team.get(
                    "players",
                    {}
                )

                for player in players.values():

                    person = player.get(
                        "person",
                        {}
                    )

                    player_id = person.get(
                        "id"
                    )

                    position = player.get(
                        "position",
                        {}
                    )

                    position_code = str(
                        position.get(
                            "code",
                            ""
                        )
                    )

                    position_abbreviation = str(
                        position.get(
                            "abbreviation",
                            ""
                        )
                    ).upper()

                    # MLB StatsAPI: PH / position code 11
                    if (
                        position_code == "11"
                        or
                        position_abbreviation == "PH"
                    ):
                        if player_id is not None:
                            pinch_hitter_ids.add(
                                str(player_id)
                            )

        except Exception as e:

            print(
                f"Warning: could not check "
                f"game {game_pk}: {e}"
            )

        time.sleep(
            request_pause
        )

    # --------------------------------------------------------
    # Match MLBAM batter IDs
    # --------------------------------------------------------

    pa["batter_mlbam"] = (
        pd.to_numeric(
            pa["batter_mlbam"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    pinch_hitter_ids = set(
        str(x)
        for x in pinch_hitter_ids
    )

    pa["is_pinch_hitter"] = (
        pa["batter_mlbam"]
        .isin(pinch_hitter_ids)
    )

    pa["pinch_hitter_source"] = np.where(
        pa["is_pinch_hitter"],
        "MLB_StatsAPI",
        "MLB_StatsAPI_checked"
    )

    print(
        f"Identified "
        f"{pa['is_pinch_hitter'].sum():,} "
        f"pinch-hit PAs."
    )

    return pa

In [26]:
CURRENT_SEASON_START = date(
    CURRENT_YEAR,
    3,
    1
)
current_pa = update_current_season()

Existing current-season PAs: 187,235
Refreshing Statcast 2026-09-18 -> 2026-10-13
This is a large query, it may take a moment to complete


100%|█████████████████████████████████████████████████████████████████████████████| 26/26 [00:11<00:00,  2.17it/s]
C:\Users\blobf\anaconda3\Lib\site-packages\pybaseball\statcast.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_data = pd.concat(dataframe_list, axis=0).convert_dtypes(convert_string=False)


Downloaded 47,789 pitch rows.
Checking pinch hitters...
Identified 1,310 pinch-hit PAs.

CURRENT SEASON UPDATED
Total PAs: 196,037
First date: 2026-03-15 00:00:00
Last date: 2026-10-12 00:00:00
Saved: C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_data\pa\season=2026\pa_2026.parquet


## 7. Create hitter and pitcher observations

Do not duplicate the raw PA in the source table. Instead, create an analytical view in which the same PA is viewed once from the hitter's perspective and once from the pitcher's perspective. A positive `player_value` is therefore always good for the player.

In [28]:
# ============================================================
# DUCKDB PA DATABASE
# ============================================================
con = duckdb.connect(
    str(BASE / "mlb_clutch.duckdb")
)

# Your PA files are inside season=YYYY folders.
PA_GLOB = str(
    PA_DIR / "**" / "pa_*.parquet"
)

print("Looking for PA files:")
print(PA_GLOB)

# ------------------------------------------------------------
# Find PA files
# ------------------------------------------------------------

files = con.sql(
    f"""
    SELECT file
    FROM glob('{PA_GLOB}')
    ORDER BY file
    """
).df()

print()
print(
    f"Found {len(files):,} PA files."
)

print(
    files.head(10)
)

# ------------------------------------------------------------
# Stop if no files exist
# ------------------------------------------------------------

if len(files) == 0:

    raise FileNotFoundError(
        "No PA parquet files were found. "
        "You need to build at least one season first."
    )

# ------------------------------------------------------------
# Create PA view
# ------------------------------------------------------------

con.execute(
    f"""
    CREATE OR REPLACE VIEW pa AS

    SELECT *
    FROM read_parquet(
        '{PA_GLOB}',
        union_by_name = true
    )
    """
)

print()
print("PA VIEW CREATED")

# ------------------------------------------------------------
# Show actual columns
# ------------------------------------------------------------

columns = con.sql(
    """
    DESCRIBE pa
    """
).df()

print()
print("PA columns:")

print(
    columns[
        [
            "column_name",
            "column_type"
        ]
    ].to_string(index=False)
)

Looking for PA files:
C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_data\pa\**\pa_*.parquet

Found 1 PA files.
                                                file
0  C:\Users\blobf\OneDrive\MLB Clutch\mlb_clutch_...

PA VIEW CREATED

PA columns:
        column_name  column_type
            game_pk       BIGINT
      at_bat_number       BIGINT
          game_date TIMESTAMP_NS
       pitch_number       BIGINT
             batter       BIGINT
            pitcher       BIGINT
             inning       BIGINT
      inning_topbot      VARCHAR
       outs_when_up       BIGINT
              on_1b       BIGINT
              on_2b       BIGINT
              on_3b       BIGINT
         home_score       BIGINT
         away_score       BIGINT
          bat_score       BIGINT
          fld_score       BIGINT
          pa_result      VARCHAR
     pa_description      VARCHAR
      delta_run_exp       DOUBLE
 delta_home_win_exp       DOUBLE
             season       BIGINT
       batter_mlbam      VA

## 8. Player-level clutch statistic

The first research version should report **three related statistics**:

1. `unweighted_value_per_pa` — baseline performance.
2. `importance_weighted_value_per_pa` — performance after giving more influence to important situations.
3. `clutch_differential` — weighted performance minus baseline.

The third quantity is the most directly aligned with your idea of 'clutch': it measures whether a player's performance changes positively or negatively when the situation becomes more important.

In [30]:
# ============================================================
# PLAYER-LEVEL CLUTCH STATISTIC
# SELF-CONTAINED VERSION
# ============================================================

import pandas as pd
import numpy as np
import requests
from io import BytesIO


# ------------------------------------------------------------
# 1. Make sure AUX_DIR exists
# ------------------------------------------------------------

try:
    AUX_DIR
except NameError:
    AUX_DIR = Path.cwd() / "mlb_clutch_data" / "aux"
    AUX_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 2. Load Chadwick Register if it does not already exist
# ------------------------------------------------------------

if "register" not in globals():

    print("Chadwick Register not loaded.")
    print("Downloading it now...")

    CHADWICK_BASE = (
        "https://raw.githubusercontent.com/"
        "chadwickbureau/register/master/data/"
    )

    parts = []

    for shard in "0123456789abcdef":

        url = (
            CHADWICK_BASE
            + f"people-{shard}.csv"
        )

        print(
            f"Downloading people-{shard}.csv..."
        )

        response = requests.get(
            url,
            timeout=180
        )

        response.raise_for_status()

        part = pd.read_csv(
            BytesIO(response.content),
            low_memory=False
        )

        parts.append(part)

    register = pd.concat(
        parts,
        ignore_index=True
    )

    print(
        f"Loaded {len(register):,} register records."
    )

else:

    print(
        "Using existing Chadwick Register."
    )


# ------------------------------------------------------------
# 3. Verify the required Chadwick columns
# ------------------------------------------------------------

required_columns = [
    "key_uuid",
    "key_mlbam",
    "key_retro",
    "name_first",
    "name_last"
]

missing_columns = [
    c
    for c in required_columns
    if c not in register.columns
]

if missing_columns:

    print(
        "Missing Chadwick columns:"
    )

    print(
        missing_columns
    )

    print(
        "Available columns:"
    )

    print(
        register.columns.tolist()
    )

    raise KeyError(
        f"Chadwick Register is missing: "
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# 4. Normalize Chadwick player IDs
# ------------------------------------------------------------

def normalize_id(series):

    return (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )


register["key_mlbam"] = normalize_id(
    register["key_mlbam"]
)

register["key_retro"] = (
    register["key_retro"]
    .astype("string")
)

register["key_uuid"] = (
    register["key_uuid"]
    .astype("string")
)


# ------------------------------------------------------------
# 5. Create player names
# ------------------------------------------------------------

register["player_name"] = (
    register["name_first"]
    .fillna("")
    .astype("string")
    .str.strip()

    + " "

    +

    register["name_last"]
    .fillna("")
    .astype("string")
    .str.strip()
)

register["player_name"] = (
    register["player_name"]
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)


# ------------------------------------------------------------
# 6. Create MLBAM lookup
# ------------------------------------------------------------

mlbam_map = (
    register[
        [
            "key_mlbam",
            "key_uuid",
            "player_name"
        ]
    ]
    .dropna(
        subset=["key_mlbam"]
    )
    .drop_duplicates(
        subset=["key_mlbam"]
    )
    .rename(
        columns={
            "key_mlbam": "mlbam_id",
            "key_uuid": "person_uuid"
        }
    )
)

print(
    f"MLBAM player lookup: "
    f"{len(mlbam_map):,} players"
)


# ------------------------------------------------------------
# 7. Make sure DuckDB connection exists
# ------------------------------------------------------------

try:
    con
except NameError:

    con = duckdb.connect(
        str(
            Path.cwd()
            / "mlb_clutch_data"
            / "mlb_clutch.duckdb"
        )
    )


# ------------------------------------------------------------
# 8. Make sure the PA view exists
# ------------------------------------------------------------

try:

    con.sql(
        "SELECT COUNT(*) FROM pa"
    )

except Exception:

    PA_GLOB = str(
        Path.cwd()
        / "mlb_clutch_data"
        / "pa"
        / "**"
        / "pa_*.parquet"
    )

    print(
        "Rebuilding PA view..."
    )

    con.execute(
        f"""
        CREATE OR REPLACE VIEW pa AS

        SELECT *

        FROM read_parquet(
            '{PA_GLOB}',
            union_by_name = true
        )
        """
    )


# ------------------------------------------------------------
# 9. Register the player lookup with DuckDB
# ------------------------------------------------------------

con.register(
    "mlbam_lookup",
    mlbam_map
)


# ------------------------------------------------------------
# 10. Rebuild player_roles
# ------------------------------------------------------------

con.execute(
    """
    CREATE OR REPLACE VIEW player_roles AS

    SELECT

        pa.season,
        pa.game_date,
        pa.game_id,

        pa.inning,
        pa.top_bot,
        pa.outs_pre,

        pa.run_diff,
        pa.base_state,
        pa.risp,
        pa.is_pinch_hitter,

        pa.pa_result,

        pa.run_value,

        COALESCE(
            pa.leverage_index,
            1.0
        ) AS leverage_index,

        pa.situation_key,

        pa.pbp_quality,
        pa.data_source,

        'hitter' AS role,

        COALESCE(
            pa.batter_uuid,
            b.person_uuid
        ) AS person_uuid,

        pa.batter_mlbam AS player_mlbam,

        COALESCE(
            pa.batter_name,
            b.player_name
        ) AS player_name,

        pa.run_value AS player_value

    FROM pa

    LEFT JOIN mlbam_lookup b

        ON TRY_CAST(
            pa.batter_mlbam AS BIGINT
        )
        =
        TRY_CAST(
            b.mlbam_id AS BIGINT
        )


    UNION ALL


    SELECT

        pa.season,
        pa.game_date,
        pa.game_id,

        pa.inning,
        pa.top_bot,
        pa.outs_pre,

        pa.run_diff,
        pa.base_state,
        pa.risp,
        pa.is_pinch_hitter,

        pa.pa_result,

        pa.run_value,

        COALESCE(
            pa.leverage_index,
            1.0
        ) AS leverage_index,

        pa.situation_key,

        pa.pbp_quality,
        pa.data_source,

        'pitcher' AS role,

        COALESCE(
            pa.pitcher_uuid,
            p.person_uuid
        ) AS person_uuid,

        pa.pitcher_mlbam AS player_mlbam,

        COALESCE(
            pa.pitcher_name,
            p.player_name
        ) AS player_name,

        -pa.run_value AS player_value

    FROM pa

    LEFT JOIN mlbam_lookup p

        ON TRY_CAST(
            pa.pitcher_mlbam AS BIGINT
        )
        =
        TRY_CAST(
            p.mlbam_id AS BIGINT
        )
    """
)


# ------------------------------------------------------------
# 11. Check names BEFORE calculating the metric
# ------------------------------------------------------------

check = con.sql(
    """
    SELECT

        role,

        COUNT(*) AS observations,

        COUNT(player_name)
            AS named_observations,

        COUNT(*)
        -
        COUNT(player_name)
            AS missing_names

    FROM player_roles

    GROUP BY role

    ORDER BY role
    """
).df()

print()
print("PLAYER NAME CHECK")
print(check)


# ------------------------------------------------------------
# 12. Calculate player-level clutch metric
# ------------------------------------------------------------

summary = con.sql(
    """
    SELECT

        person_uuid,

        player_mlbam,

        player_name,

        role,

        COUNT(*) AS pa,

        AVG(player_value)
            AS unweighted_value_per_pa,

        SUM(
            player_value
            *
            leverage_index
        )
        /
        NULLIF(
            SUM(leverage_index),
            0
        )
            AS importance_weighted_value_per_pa,

        (
            SUM(
                player_value
                *
                leverage_index
            )
            /
            NULLIF(
                SUM(leverage_index),
                0
            )
        )
        -
        AVG(player_value)
            AS clutch_differential,

        SUM(
            player_value
            *
            leverage_index
        )
            AS total_weighted_value,

        AVG(leverage_index)
            AS mean_leverage

    FROM player_roles

    WHERE player_name IS NOT NULL

    GROUP BY

        person_uuid,

        player_mlbam,

        player_name,

        role
    """
).df()


# ------------------------------------------------------------
# 13. Save
# ------------------------------------------------------------

summary.to_csv(
    AUX_DIR
    / "player_clutch_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 14. Display
# ------------------------------------------------------------

print()
print(
    "========================================"
)

print(
    "PLAYER-LEVEL CLUTCH STATISTIC"
)

print(
    "========================================"
)

print(
    f"Player-role records: "
    f"{len(summary):,}"
)

summary.head(20)

Using existing Chadwick Register.
MLBAM player lookup: 135,208 players

PLAYER NAME CHECK
      role  observations  named_observations  missing_names
0   hitter        196037              196037              0
1  pitcher        196037              196037              0

PLAYER-LEVEL CLUTCH STATISTIC
Player-role records: 2,603


,person_uuid,player_mlbam,player_name,role,pa,unweighted_value_per_pa,importance_weighted_value_per_pa,clutch_differential,total_weighted_value,mean_leverage
0,bd8e6213-614e-444c-82be-2ddf956d12da,571657,Kyle Farmer,hitter,37,-0.128108,-0.128108,0.0,-4.740,1.0
1,dae6256a-d035-47b8-84a3-200efcb5544e,665561,Rafael Marchán,hitter,109,-0.099037,-0.099037,0.0,-10.795,1.0
2,88874cc0-3ecc-477b-b3f9-0164167829ea,644433,Chadwick Tromp,hitter,54,-0.106463,-0.106463,0.0,-5.749,1.0
3,9754ea1e-0fa1-40ff-864c-355f5f61e78c,701675,Nathan Church,hitter,486,-0.009891,-0.009891,0.0,-4.807,1.0
4,f0e8fd62-774b-401d-b8d3-17a69b517cff,514888,José Altuve,hitter,596,0.001097,0.001097,0.0,0.654,1.0
5,600806fb-f653-42bd-b659-ca3466c59c6f,702315,Bennett Hostetler,hitter,1,-0.262000,-0.262000,0.0,-0.262,1.0
6,4d1d549c-435a-470f-a96c-db49150e412e,703563,PJ Morlando,hitter,2,-0.210000,-0.210000,0.0,-0.420,1.0
7,9520bd9a-6f78-432a-9280-f0301b9e9486,687462,Spencer Horwitz,hitter,469,-0.005134,-0.005134,0.0,-2.408,1.0
8,dcfb7ca5-c0e6-4aa0-9e8e-f1d5699925c0,681481,Kerry Carpenter,hitter,283,0.011106,0.011106,0.0,3.143,1.0
9,7caae4be-c392-4aa4-99ac-061769ee23b5,680474,Max Schuemann,hitter,108,0.002935,0.002935,0.0,0.317,1.0


In [31]:
# ============================================================
# PA-LEVEL SEASON-TO-DATE CLUTCH RATING
# ============================================================
#
# Each row remains one player observation for one PA.
#
# The clutch rating is calculated using ONLY the player's
# performance up to and including that PA in that season.
#
# This prevents future-season / future-game information
# from leaking into earlier PAs.
# ============================================================


# ------------------------------------------------------------
# 1. Create a chronological PA sequence
# ------------------------------------------------------------
#
# We use the information currently stored in player_roles.
#
# game_date + game_id + inning + top_bot + outs_pre provide
# the chronological structure available in the current file.
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW player_roles_ordered AS

SELECT

    *,

    ROW_NUMBER() OVER (
        PARTITION BY
            person_uuid,
            role,
            season

        ORDER BY
            game_date,
            game_id,
            inning,
            CASE
                WHEN top_bot = '0' THEN 0
                WHEN top_bot = '1' THEN 1
                ELSE TRY_CAST(top_bot AS INTEGER)
            END,
            outs_pre,
            game_id
    ) AS season_pa_number

FROM player_roles

WHERE player_name IS NOT NULL
""")


# ------------------------------------------------------------
# 2. Calculate running season statistics
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW player_pa_clutch AS

WITH running AS (

    SELECT

        *,

        -- -----------------------------------------------
        -- Number of PAs this player has had this season
        -- up through this PA
        -- -----------------------------------------------

        season_pa_number
            AS season_to_date_pa,


        -- -----------------------------------------------
        -- Cumulative unweighted value
        -- -----------------------------------------------

        SUM(player_value) OVER (

            PARTITION BY
                person_uuid,
                role,
                season

            ORDER BY
                season_pa_number

            ROWS BETWEEN
                UNBOUNDED PRECEDING
                AND CURRENT ROW

        ) AS season_to_date_total_value,


        -- -----------------------------------------------
        -- Cumulative weighted value
        -- -----------------------------------------------

        SUM(
            player_value
            * leverage_index
        ) OVER (

            PARTITION BY
                person_uuid,
                role,
                season

            ORDER BY
                season_pa_number

            ROWS BETWEEN
                UNBOUNDED PRECEDING
                AND CURRENT ROW

        ) AS season_to_date_weighted_total,


        -- -----------------------------------------------
        -- Cumulative importance weight
        -- -----------------------------------------------

        SUM(
            leverage_index
        ) OVER (

            PARTITION BY
                person_uuid,
                role,
                season

            ORDER BY
                season_pa_number

            ROWS BETWEEN
                UNBOUNDED PRECEDING
                AND CURRENT ROW

        ) AS season_to_date_weight,


        -- -----------------------------------------------
        -- Cumulative number of observations
        -- -----------------------------------------------

        COUNT(*) OVER (

            PARTITION BY
                person_uuid,
                role,
                season

            ORDER BY
                season_pa_number

            ROWS BETWEEN
                UNBOUNDED PRECEDING
                AND CURRENT ROW

        ) AS season_to_date_count

    FROM player_roles_ordered
)

SELECT

    *,

    -- ====================================================
    -- SEASON-TO-DATE ORDINARY PERFORMANCE
    -- ====================================================

    season_to_date_total_value
    /
    NULLIF(
        season_to_date_count,
        0
    )
        AS season_to_date_value_per_pa,


    -- ====================================================
    -- SEASON-TO-DATE IMPORTANCE-WEIGHTED PERFORMANCE
    -- ====================================================

    season_to_date_weighted_total
    /
    NULLIF(
        season_to_date_weight,
        0
    )
        AS season_to_date_weighted_value_per_pa,


    -- ====================================================
    -- CLUTCH DIFFERENTIAL AT THIS PA
    --
    -- weighted performance - ordinary performance
    -- ====================================================

    (
        season_to_date_weighted_total
        /
        NULLIF(
            season_to_date_weight,
            0
        )
    )
    -
    (
        season_to_date_total_value
        /
        NULLIF(
            season_to_date_count,
            0
        )
    )
        AS clutch_differential,


    -- ====================================================
    -- HUMAN-READABLE CLUTCH RATING
    --
    -- This is the same value as clutch_differential,
    -- explicitly named as the player's rating at this
    -- point in the season.
    -- ====================================================

    (
        season_to_date_weighted_total
        /
        NULLIF(
            season_to_date_weight,
            0
        )
    )
    -
    (
        season_to_date_total_value
        /
        NULLIF(
            season_to_date_count,
            0
        )
    )
        AS season_clutch_rating

FROM running
""")


print(
    "PA-level clutch rating table created."
)


# ------------------------------------------------------------
# 3. Display the new columns
# ------------------------------------------------------------

pa_clutch_sample = con.sql("""
SELECT

    season,
    game_date,
    game_id,

    inning,
    top_bot,
    outs_pre,
    run_diff,
    base_state,

    player_name,
    role,

    player_value,
    leverage_index,

    season_to_date_pa,

    season_to_date_value_per_pa,

    season_to_date_weighted_value_per_pa,

    clutch_differential,

    season_clutch_rating

FROM player_pa_clutch

ORDER BY
    player_name,
    role,
    season,
    season_pa_number

LIMIT 50
""").df()


display(pa_clutch_sample)

PA-level clutch rating table created.


,season,game_date,game_id,inning,top_bot,outs_pre,run_diff,base_state,player_name,role,player_value,leverage_index,season_to_date_pa,season_to_date_value_per_pa,season_to_date_weighted_value_per_pa,clutch_differential,season_clutch_rating
0,2026,2026-03-16,831834,9,0,0,-3,000,A. J. Causey,pitcher,0.260,1.0,1,0.260000,0.260000,0.0,0.0
1,2026,2026-03-16,831834,9,0,1,-3,000,A. J. Causey,pitcher,-0.453,1.0,2,-0.096500,-0.096500,0.0,0.0
2,2026,2026-03-16,831834,9,0,1,-3,110,A. J. Causey,pitcher,0.297,1.0,3,0.034667,0.034667,0.0,0.0
3,2026,2026-03-16,831834,9,0,2,-3,001,A. J. Causey,pitcher,-0.098,1.0,4,0.001500,0.001500,0.0,0.0
4,2026,2026-03-16,831834,9,0,2,-3,101,A. J. Causey,pitcher,0.210,1.0,5,0.043200,0.043200,0.0,0.0
5,2026,2026-05-12,823632,2,1,1,-2,110,A. J. Ewing,hitter,0.294,1.0,1,0.294000,0.294000,0.0,0.0
6,2026,2026-05-12,823632,4,1,0,0,000,A. J. Ewing,hitter,-0.303,1.0,2,-0.004500,-0.004500,0.0,0.0
7,2026,2026-05-12,823632,6,1,1,1,000,A. J. Ewing,hitter,0.342,1.0,3,0.111000,0.111000,0.0,0.0
8,2026,2026-05-12,823632,7,1,2,4,100,A. J. Ewing,hitter,1.339,1.0,4,0.418000,0.418000,0.0,0.0
9,2026,2026-05-12,823632,8,1,2,7,111,A. J. Ewing,hitter,0.317,1.0,5,0.397800,0.397800,0.0,0.0


In [32]:
summary = con.sql("""
SELECT person_uuid, player_mlbam, player_name, role,
       COUNT(*) AS pa,
       AVG(player_value) AS unweighted_value_per_pa,
       SUM(player_value * leverage_index) / NULLIF(SUM(leverage_index),0) AS importance_weighted_value_per_pa,
       SUM(player_value * leverage_index) / NULLIF(SUM(leverage_index),0) - AVG(player_value) AS clutch_differential,
       SUM(player_value * leverage_index) AS total_weighted_value,
       AVG(leverage_index) AS mean_leverage
FROM player_roles
WHERE player_name IS NOT NULL
GROUP BY 1,2,3,4
""").df()

summary.to_csv(SUMMARY_DIR / 'player_clutch_summary.csv', index=False)
summary.head(20)


,person_uuid,player_mlbam,player_name,role,pa,unweighted_value_per_pa,importance_weighted_value_per_pa,clutch_differential,total_weighted_value,mean_leverage
0,56ba1da7-6d14-4467-9a61-1c23930afb1e,640448,Kyle Finnegan,pitcher,275,-0.040385,-0.040385,0.0,-11.106,1.0
1,9b0770f3-9aa9-4e1f-970f-f24b0da3890d,664918,Kyle Keller,pitcher,15,0.016000,0.016000,0.0,0.240,1.0
2,e7b609cb-136e-49be-b25c-e65a3c514938,686560,Gavin Collyer,pitcher,123,-0.013821,-0.013821,0.0,-1.700,1.0
3,f0f360a6-b07d-45e1-8348-e231ae388680,642528,Jonathan Loáisiga,pitcher,274,0.041307,0.041307,0.0,11.318,1.0
4,a30fe373-56d6-4837-992f-98b3420d3458,805299,Brandyn Garcia,pitcher,208,0.064293,0.064293,0.0,13.373,1.0
5,a4f690fb-2a38-4ab0-b02f-0ec55d34bce0,676502,Roman Phansalkar,pitcher,2,-0.087500,-0.087500,0.0,-0.175,1.0
6,cd538f22-2eab-430d-8793-81574ce7c72f,678020,Seth Halvorsen,pitcher,155,-0.074213,-0.074213,0.0,-11.503,1.0
7,8723f0e4-0e6c-4e4b-9187-a6f7c61c0365,666374,Matt Brash,pitcher,79,0.063722,0.063722,0.0,5.034,1.0
8,0b5b13ad-4fd2-455a-b12d-7f995ca7df7c,668674,Lucas Erceg,pitcher,245,-0.053931,-0.053931,0.0,-13.213,1.0
9,f1d7dbf9-abe1-486b-883a-c3575d63fb1b,676428,Brant Hurter,pitcher,153,0.051745,0.051745,0.0,7.917,1.0


## 9. Every player's average in every situation

This table is the research dataset behind the final metric. It preserves the exact situation dimensions so you can later test alternative definitions of 'important.'

In [34]:
# ============================================================
# PLAYER × SITUATION SUMMARY
# ============================================================

player_situation = con.sql("""
SELECT
    person_uuid,
    player_mlbam,
    player_name,
    role,
    season,

    situation_key,

    inning,
    top_bot,
    outs_pre,

    run_diff,
    base_state,
    risp,
    is_pinch_hitter,

    COUNT(*) AS pa,

    -- Average performance in this situation
    AVG(player_value)
        AS avg_player_value,

    -- Total performance in this situation
    SUM(player_value)
        AS total_player_value,

    -- Importance of the situation
    AVG(leverage_index)
        AS situation_leverage,

    -- Performance weighted by situation importance
    SUM(
        player_value * leverage_index
    )
    /
    NULLIF(
        SUM(leverage_index),
        0
    )
        AS weighted_situation_value

FROM player_roles

WHERE player_name IS NOT NULL

GROUP BY
    person_uuid,
    player_mlbam,
    player_name,
    role,
    season,
    situation_key,
    inning,
    top_bot,
    outs_pre,
    run_diff,
    base_state,
    risp,
    is_pinch_hitter

ORDER BY
    player_name,
    role,
    season,
    situation_leverage DESC
""").df()


# Save the detailed situation table
player_situation.to_csv(
    SUMMARY_DIR /
    "player_clutch_by_situation.csv",
    index=False
)


print(
    f"Player × situation rows: "
    f"{len(player_situation):,}"
)

display(
    player_situation.head(20)
)

Player × situation rows: 322,477


,person_uuid,player_mlbam,player_name,role,season,situation_key,inning,top_bot,outs_pre,run_diff,base_state,risp,is_pinch_hitter,pa,avg_player_value,total_player_value,situation_leverage,weighted_situation_value
0,86215cd4-fc44-46be-8f5e-f87a998c46d6,822067,A. J. Causey,pitcher,2026,9|-3|0|1|110,9,0,1,-3,110,True,False,1,0.297000,0.297,1.0,0.297000
1,86215cd4-fc44-46be-8f5e-f87a998c46d6,822067,A. J. Causey,pitcher,2026,9|-3|0|1|000,9,0,1,-3,000,False,False,1,-0.453000,-0.453,1.0,-0.453000
2,86215cd4-fc44-46be-8f5e-f87a998c46d6,822067,A. J. Causey,pitcher,2026,9|-3|0|0|000,9,0,0,-3,000,False,False,1,0.260000,0.260,1.0,0.260000
3,86215cd4-fc44-46be-8f5e-f87a998c46d6,822067,A. J. Causey,pitcher,2026,9|-3|0|2|001,9,0,2,-3,001,True,True,1,-0.098000,-0.098,1.0,-0.098000
4,86215cd4-fc44-46be-8f5e-f87a998c46d6,822067,A. J. Causey,pitcher,2026,9|-3|0|2|101,9,0,2,-3,101,True,True,1,0.210000,0.210,1.0,0.210000
5,22839689-206d-4698-8be7-ca9762260a10,805999,A. J. Ewing,hitter,2026,5|1|1|0|000,5,1,0,1,000,False,False,2,-0.377000,-0.754,1.0,-0.377000
6,22839689-206d-4698-8be7-ca9762260a10,805999,A. J. Ewing,hitter,2026,3|-1|0|0|010,3,0,0,-1,010,True,False,1,0.486000,0.486,1.0,0.486000
7,22839689-206d-4698-8be7-ca9762260a10,805999,A. J. Ewing,hitter,2026,9|3|0|0|000,9,0,0,3,000,False,False,1,-0.347000,-0.347,1.0,-0.347000
8,22839689-206d-4698-8be7-ca9762260a10,805999,A. J. Ewing,hitter,2026,7|-3|1|2|111,7,1,2,-3,111,True,False,1,-0.261000,-0.261,1.0,-0.261000
9,22839689-206d-4698-8be7-ca9762260a10,805999,A. J. Ewing,hitter,2026,5|-4|1|0|110,5,1,0,-4,110,True,False,1,-0.255000,-0.255,1.0,-0.255000


## 10. Example matchup queries


In [36]:
# Hitter outcomes
con.sql("SELECT player_name, pa_result, COUNT(*) AS pa FROM player_roles WHERE role='hitter' GROUP BY 1,2 ORDER BY player_name, pa DESC").df().head(50)

# Pitcher outcomes allowed
con.sql("SELECT player_name, pa_result, COUNT(*) AS pa FROM player_roles WHERE role='pitcher' GROUP BY 1,2 ORDER BY player_name, pa DESC").df().head(50)

# Hitter vs pitcher matchup
con.sql("SELECT batter_name, pitcher_name, pa_result, COUNT(*) AS pa, AVG(run_value) AS avg_run_value FROM pa GROUP BY 1,2,3 ORDER BY pa DESC").df().head(100)


,batter_name,pitcher_name,pa_result,pa,avg_run_value
0,<NA>,<NA>,field_out,78604,-0.254518
1,<NA>,<NA>,strikeout,43377,-0.221259
2,<NA>,<NA>,single,27739,0.445441
3,<NA>,<NA>,walk,16909,0.234568
4,<NA>,<NA>,double,7997,0.723584
5,<NA>,<NA>,home_run,5954,1.530351
6,<NA>,<NA>,force_out,3608,-0.284476
7,<NA>,<NA>,grounded_into_double_play,3352,-0.315180
8,<NA>,<NA>,hit_by_pitch,2264,0.361248
9,<NA>,<NA>,sac_fly,1368,-0.156260


In [38]:
con.sql("""
SELECT
    player_name,
    role,

    COUNT(*) AS pa,

    AVG(player_value) AS average_value,

    SUM(
        player_value * leverage_index
    ) / SUM(leverage_index)
        AS weighted_value,

    (
        SUM(
            player_value * leverage_index
        ) / SUM(leverage_index)
    )
    -
    AVG(player_value)
        AS clutch_differential,

    AVG(leverage_index)
        AS average_leverage

FROM player_roles

WHERE player_name IS NOT NULL

GROUP BY
    player_name,
    role

HAVING COUNT(*) >= 100

ORDER BY
    clutch_differential DESC
""").df().sort_values(
    by="weighted_value",
    ascending=False
)

,player_name,role,pa,average_value,weighted_value,clutch_differential,average_leverage
993,Eduardo Valencia,hitter,142,0.101577,0.101577,0.0,1.0
103,Josh Hader,pitcher,177,0.094955,0.094955,0.0,1.0
692,Hagen Smith,pitcher,149,0.085805,0.085805,0.0,1.0
742,Willson Contreras,hitter,559,0.085483,0.085483,0.0,1.0
721,Joey Wiemer,hitter,102,0.074853,0.074853,0.0,1.0
...,...,...,...,...,...,...,...
677,Denzel Clarke,hitter,103,-0.108641,-0.108641,0.0,1.0
603,Sean Sullivan,pitcher,114,-0.111974,-0.111974,0.0,1.0
25,Logan Allen,pitcher,104,-0.120000,-0.120000,0.0,1.0
511,Luis Morales,pitcher,122,-0.122221,-0.122221,0.0,1.0


## Research cautions and next steps

* Retrosheet older seasons include some deduced or incomplete games; keep `pbp_quality` and run sensitivity checks.
* Statcast's `delta_run_exp` and `delta_home_win_exp` are modern fields. For historical comparisons, RE24 is more consistent and transparent.
* The live 2026 Statcast feed does not directly carry Retrosheet's explicit pinch-hit flag in the simple PA table above, so it is intentionally left `NA` rather than guessed.
* Use minimum-sample thresholds before interpreting a player's situation average.
* Do not choose the formula because it creates an attractive leaderboard. Freeze the formula before evaluating players.
* For a stronger paper, compare the metric with existing concepts such as WPA/LI and test out-of-sample stability across seasons.

### Data sources
Retrosheet: https://www.retrosheet.org/downloads/plays.html
Baseball Savant CSV documentation: https://baseballsavant.mlb.com/csv-docs
Chadwick Register: https://github.com/chadwickbureau/register
pybaseball: https://github.com/jldbc/pybaseball


## Recommended run order

1. Install packages and load the Chadwick register.
2. Test with `download_retrosheet_year(2025)`.
3. When that works, run the 1908–2025 backfill.
4. Build `re24.parquet`.
5. Build the leverage table from 2024–2025 Statcast.
6. Convert all historical seasons to PA Parquet.
7. Run `update_current_season()` whenever you want fresh 2026 data.
8. Re-create the DuckDB views and player summaries.

For a continuous project, put steps 7–8 in a scheduled Python script/GitHub Action rather than depending on an open notebook.

In [43]:
#Final leader Board Searcher

player_name = "Shohei Ohtani"

summary[
    summary["player_name"].str.contains(
        player_name, case=False, na=False
    )
].sort_values(by="clutch_rating", ascending=False)

KeyError: 'clutch_rating'

In [ ]:
#After  Every PA

player_name = "Shohei Ohtani"

con.execute("""
    SELECT *
    FROM player_pa_clutch
    WHERE LOWER(player_name) LIKE LOWER(?)
    ORDER BY season, game_date, game_id, season_pa_number
""", [f"%{player_name}%"]).df()

In [ ]:
#Season for player

con.execute("""
    SELECT *
    FROM player_pa_clutch
    WHERE LOWER(player_name) LIKE LOWER(?)
      AND season = 2025
    ORDER BY game_date, game_id, season_pa_number
""", ["%shohei ohtani%"]).df()

In [ ]:
summary.sort_values(
    by="clutch_rating",
    ascending=False
).head(20)

In [ ]:
##Most clutch At bats

most_clutch_ats = con.execute("""
    SELECT *
    FROM player_pa_clutch
    WHERE clutch_contribution IS NOT NULL
    ORDER BY clutch_contribution DESC
    LIMIT 25
""").df()

display(most_clutch_ats)

In [ ]:
#Most clutch at bats by situation adjusted run value

most_clutch_ats = con.execute("""
    SELECT *
    FROM player_pa_clutch
    WHERE run_value IS NOT NULL
    ORDER BY (run_value * leverage_index) DESC
    LIMIT 25
""").df()

display(most_clutch_ats)